# Debug: torch.tensor() Conversion Issue

This notebook isolates the hanging `torch.tensor()` conversion in `04-model_training.ipynb`.

In [ ]:
import time
import pickle
import numpy as np
import pandas as pd
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
# Load only the data we need
with open("../data/03-result/drugs_df.pkl", "rb") as f:
    drugs_df = pickle.load(f)
with open("../data/03-result/diseases_df.pkl", "rb") as f:
    diseases_df = pickle.load(f)
    
print(f"drugs_df: {drugs_df.shape}")
print(f"diseases_df: {diseases_df.shape}")

## Step 1: Prepare drug_features DataFrame

In [ ]:
drug_features = drugs_df.drop(columns=drugs_df.filter(regex="path").columns)
drug_features = drug_features.select_dtypes(include=["number", "bool"])
print(f"drug_features shape: {drug_features.shape}")
print(f"Columns: {drug_features.columns.tolist()[:10]}...")

## Step 2: Diagnose the data before tensor conversion

In [ ]:
# Check dtypes
print("=== Data types ===")
print(drug_features.dtypes.value_counts())

# Check for problematic values
print("\n=== NaN counts ===")
nan_counts = drug_features.isna().sum()
print(f"Total NaN values: {nan_counts.sum()}")
if nan_counts.sum() > 0:
    print(nan_counts[nan_counts > 0])

# Check for inf values
print("\n=== Inf counts ===")
numeric_cols = drug_features.select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(drug_features[numeric_cols]).sum()
print(f"Total Inf values: {inf_counts.sum()}")

# Check for object columns that slipped through
print("\n=== Object columns (should be none) ===")
obj_cols = drug_features.select_dtypes(include=['object']).columns
print(f"Object columns: {list(obj_cols)}")

## Step 3: Test numpy conversion first (before torch)

In [ ]:
t0 = time.time()
values = drug_features.values
print(f"df.values: {time.time()-t0:.3f}s, dtype: {values.dtype}, shape: {values.shape}")

t0 = time.time()
values_float = values.astype(np.float64)
print(f"astype(float64): {time.time()-t0:.3f}s, dtype: {values_float.dtype}")

## Step 4: Test torch.tensor() with small data first

In [ ]:
# Test with tiny array
t0 = time.time()
tiny = torch.tensor(np.random.randn(10, 10), dtype=torch.float)
print(f"Tiny tensor (10x10): {time.time()-t0:.3f}s")

# Test with same shape as drug_features
t0 = time.time()
test_arr = np.random.randn(*drug_features.shape).astype(np.float64)
test_tensor = torch.tensor(test_arr, dtype=torch.float)
print(f"Random tensor same shape {drug_features.shape}: {time.time()-t0:.3f}s")

## Step 5: The actual conversion (if above worked)

---\n## Important: Saved Models Already Exist!\n\nYou don't need to retrain the ML models. They're saved at:\n- `../results/models/time_split-models.pkl`\n- `../results/models/random_split-models.pkl`\n\nIn `04-model_training.ipynb`, set `RUN_TUNING = False` to load existing models instead of retraining.\n\n### What the GNN section needs:\n1. `drug_features` - tensor from drugs_df numeric columns\n2. `disease_features` - tensor from diseases_df numeric columns  \n3. `all_pathways` - array of unique pathway IDs\n4. `drugs_df`, `diseases_df`, `merged_df` - for building graph edges\n\nThe hanging issue is in step 1/2 (tensor conversion), not the GNN itself.

In [ ]:
# Fill NaN first (if any)
drug_features_clean = drug_features.fillna(0)

t0 = time.time()
drug_tensor = torch.tensor(drug_features_clean.values.astype(np.float64), dtype=torch.float, device="cpu")
print(f"Drug tensor: {time.time()-t0:.3f}s, shape: {drug_tensor.shape}")

## Alternative: Use torch.from_numpy() (faster)

In [ ]:
# torch.from_numpy is generally faster than torch.tensor
t0 = time.time()
arr = drug_features_clean.values.astype(np.float32)  # float32 is usually enough
drug_tensor_v2 = torch.from_numpy(arr)
print(f"from_numpy (float32): {time.time()-t0:.3f}s, shape: {drug_tensor_v2.shape}")

---
## Summary

If the issue is found, update the main notebook with the fix.